# Data Leakage

## 1. What Is Data Leakage?

Data leakage occurs when a Machine Learning model receives information during training that would not legitimately be available when making predictions in the real world.

This can make a model appear more accurate during evaluation than it actually is.

**Example:** Suppose we build a model to predict whether a customer will leave a company. If we accidentally include a column that records whether the customer has already cancelled their subscription, the model may achieve very high accuracy.

However, this information would not be available when predicting whether an active customer will leave in the future.

The model's evaluation becomes misleading because it has access to information that gives away the answer.

## 2. Why Is Data Leakage Dangerous?

Data leakage can:

* Produce unrealistically high evaluation scores.
* Make a model appear better than it really is.
* Lead to poor predictions on new data.
* Cause incorrect business decisions.
* Hide weaknesses in the model.
* Waste time and resources during deployment.

A model with 99% accuracy is not necessarily good. We must check whether it learned genuine patterns or received information that revealed the target.

## 3. Types of Data Leakage

### A. Target Leakage

Target leakage occurs when a feature contains information that directly or indirectly reveals the target variable.

**Example: Hospital Readmission Prediction**

Suppose a hospital wants to predict whether a patient will be readmitted.

The dataset contains:

* Patient age
* Blood pressure
* Previous medical history
* Readmission status
* A field recorded only after the patient is readmitted

If the last field is used to predict readmission before it occurs, the model receives information from the future.

**Solution:** Remove features that reveal the target or would not be available at prediction time.

### B. Train-Test Contamination

Train-test contamination occurs when information from the test set influences the model's training or preprocessing.

**Example:**

1. A dataset contains 1,000 records.
2. We calculate the mean and standard deviation using all 1,000 records.
3. We standardize the entire dataset.
4. We split it into training and testing sets.

The preprocessing step has already used information from the test set.

**Correct approach:**

1. Split the data first.
2. Fit the scaler using training data only.
3. Transform the training data.
4. Transform the test data using the same fitted scaler.

### C. Temporal Leakage

Temporal leakage occurs when information from the future is used to predict an outcome in the past.

**Example:** A sales forecasting model uses next month's sales information to predict this month's sales.

This is not a valid prediction setup because future information would not be available at the time of prediction.

**Solution:** Respect the timeline and use only information available before or at the prediction time.

### D. Preprocessing Leakage

Preprocessing leakage occurs when data preparation steps use information from validation or test data before the model is evaluated.

Examples include:

* Scaling the complete dataset before splitting.
* Imputing missing values using statistics calculated from the complete dataset.
* Selecting features using the complete dataset.
* Performing oversampling or SMOTE before splitting the dataset.

**Solution:** Split the data first and fit preprocessing steps on training data only.

## 4. Common Causes of Data Leakage

| Cause                              | Why It Is a Problem                                             | Correct Approach                                   |
| ---------------------------------- | --------------------------------------------------------------- | -------------------------------------------------- |
| Scaling before splitting           | Test data influences scaling parameters                         | Fit the scaler on training data only               |
| Imputing before splitting          | Test data influences imputation values                          | Fit the imputer on training data only              |
| Feature selection before splitting | Test data influences which features are selected                | Select features using training data only           |
| SMOTE before splitting             | Synthetic samples may incorporate information from test samples | Split first and apply SMOTE only to training data  |
| Using future information           | The model receives information unavailable at prediction time   | Respect the prediction timeline                    |
| Using target-derived features      | A feature may reveal the answer                                 | Remove invalid or unavailable features             |
| Tuning repeatedly on test results  | The test set influences model decisions                         | Use validation data or cross-validation for tuning |

## 5. Correct Order of Machine Learning Preprocessing

A safe workflow is:

1. Define the target and prediction task.
2. Identify features that would be available at prediction time.
3. Split the data into training, validation, and testing sets.
4. Fit preprocessing steps using training data only.
5. Transform the training and validation data using the fitted transformations.
6. Train the model using training data.
7. Use validation data for model selection and tuning.
8. Keep the test set separate during model development.
9. Evaluate the selected model on the test set at the end.

For a final model that is retrained using training and validation data, fit the required preprocessing steps again using only that combined training-and-validation data before the final test evaluation.

## 6. How Does a Scikit-Learn Pipeline Help?

A `Pipeline` combines preprocessing steps and a Machine Learning model into one workflow.

For example:

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

When we fit the pipeline on training data:

* The imputer learns the mean from the training data.
* The scaler learns the training data's mean and standard deviation.
* The model learns from the transformed training data.

When we predict on test data, the already-fitted imputer and scaler transform it without learning new statistics from the test set.

For feature selection, the selection step can also be included in the pipeline.

For oversampling methods such as SMOTE, use an appropriate imbalanced-learn pipeline so resampling happens within training folds rather than before cross-validation.

**Important:** A pipeline cannot automatically prevent every type of leakage. We must still ensure that the input features are valid, respect time, and do not reveal the target.

## 7. How Can We Detect Data Leakage?

Ask these questions:

* Does any feature directly reveal the target?
* Would this feature be available when making a real prediction?
* Were preprocessing statistics calculated using test data?
* Was feature selection performed before splitting?
* Was oversampling applied before splitting?
* Did future information enter the training data?
* Were test results repeatedly used to tune the model?
* Is the evaluation score suspiciously high compared with realistic expectations?

A high score does not prove leakage, but it is a reason to investigate the data and evaluation process.

## 8. Data Leakage vs Overfitting

**Data leakage:**
The model receives information that should not have been available during training or evaluation.

**Overfitting:**
The model learns patterns that are too specific to its training data and does not generalize well to unseen data.

Both can lead to poor real-world performance, but they are different problems.

## 9. Key Takeaways

* Data leakage makes model evaluation unreliable.
* Target leakage occurs when features reveal the outcome.
* Train-test contamination occurs when test information influences training or preprocessing.
* Temporal leakage occurs when future information is used to predict the past.
* Split the data before fitting preprocessing transformations.
* Fit imputers, scalers, and feature selectors on training data only.
* Apply oversampling only to training data.
* Use validation data or cross-validation for model tuning.
* Keep the test set separate until final evaluation.
* A pipeline helps prevent preprocessing leakage but does not solve every leakage problem.


# Demonstration of the Concept – Data Leakage

## Real-World Example: Customer Churn Prediction

Imagine a telecommunications company wants to predict whether an active customer will leave its service.

The dataset contains these features:

| Feature           | Description                                        | Safe to Use?                                  |
| ----------------- | -------------------------------------------------- | --------------------------------------------- |
| Tenure            | Number of months the customer has used the service | Usually yes                                   |
| Monthly Charges   | Current monthly bill                               | Usually yes                                   |
| Contract Type     | Customer's current contract                        | Usually yes                                   |
| Support Calls     | Support calls recorded before prediction time      | Usually yes                                   |
| Cancellation Date | Date the customer actually cancelled               | Usually no, if predicting before cancellation |
| Churn             | Whether the customer left                          | Target variable                               |

The features we use must be available at the time the prediction is made.

### Scenario 1: Target Leakage

Suppose the company includes `Cancellation Date` when predicting whether an active customer will leave.

Customers who have already cancelled have a cancellation date, so the model can use this information to identify the target.

The model may achieve an unusually high score, but it will not be useful for predicting churn before cancellation.

**Solution:** Remove the cancellation date from the prediction features when it reveals the outcome.

### Scenario 2: Preprocessing Leakage

Suppose we have 1,000 customer records and want to standardize monthly charges.

An incorrect approach is:

1. Calculate the mean and standard deviation from all 1,000 records.
2. Standardize all records.
3. Split the data into training and testing sets.

The test set has influenced the scaling parameters.

The correct approach is:

1. Split the data into training and testing sets.
2. Fit `StandardScaler` on the training features.
3. Transform the training features.
4. Transform the test features using the same fitted scaler.

This prevents the test data from influencing the scaling parameters.

### Scenario 3: Leakage During Imbalanced-Data Handling

Suppose only 5% of the customers belong to the churn class.

An incorrect approach is to apply SMOTE to the entire dataset and then split it.

Synthetic samples may be generated using information from records that later appear in the test set.

**Correct approach:**

1. Split the original data.
2. Apply SMOTE only to the training data.
3. Train the model on the resampled training data.
4. Evaluate the model on the untouched test data.

### Scenario 4: Temporal Leakage

Suppose the company wants to predict next month's customer churn.

Using a feature recorded only after the customer cancels would reveal future information.

**Solution:** Use only information available at the prediction date and evaluate the model using a split that respects time when the prediction task requires it.

## Conclusion

Data leakage can make a Machine Learning model appear highly accurate without actually learning useful patterns for future predictions.

To prevent it, use valid features, split the data correctly, fit preprocessing steps on training data only, and keep the test set separate until final evaluation.


In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# --------------------------------------------------
# 1. Create a sample customer churn dataset
# --------------------------------------------------

data = pd.DataFrame({
    "Tenure": [
        2, 5, 8, 10, 12, 15, 18, 20, 22, 25,
        28, 30, 32, 35, 38, 40, 42, 45, 48, 50,
        3, 6, 9, 11, 14, 16, 19, 21, 24, 27,
        29, 31, 34, 36, 39, 41, 44, 46, 49, 52
    ],
    "MonthlyCharges": [
        90, 85, 80, 75, 70, 65, 60, 55, 50, 48,
        45, 42, 40, 38, 35, 33, 30, 28, 25, 22,
        88, 82, 78, 73, 68, 63, 58, 53, 49, 46,
        43, 41, 39, 36, 34, 31, 29, 27, 24, 20
    ],
    "SupportCalls": [
        5, 4, 4, 3, 3, 2, 2, 1, 1, 1,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        5, 5, 4, 4, 3, 3, 2, 2, 1, 1,
        1, 0, 1, 0, 0, 0, 0, 0, 0, 0
    ],
    "Churn": [
        1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0
    ]
})

# This column represents information available only after
# the customer has already cancelled.
data["CancellationRecorded"] = data["Churn"]

print("Sample Dataset:")
print(data.head())

# --------------------------------------------------
# 2. Separate features and target
# --------------------------------------------------

# Correct features: information available before prediction
X = data[["Tenure", "MonthlyCharges", "SupportCalls"]]

# Target: whether the customer churned
y = data["Churn"]

# --------------------------------------------------
# 3. Split the data BEFORE preprocessing
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

# --------------------------------------------------
# 4. Build a safe preprocessing and modelling pipeline
# --------------------------------------------------

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# The scaler and model are fitted using training data only.
pipeline.fit(X_train, y_train)

# Predict using the untouched test features.
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nCorrect Workflow")
print("----------------")
print("Test Accuracy:", round(accuracy, 3))

# --------------------------------------------------
# 5. Demonstrate why target leakage is dangerous
# --------------------------------------------------

# WARNING:
# This is intentionally an invalid model setup.
# CancellationRecorded directly reveals the target.
# Do not use this approach in a real project.

X_leaky = data[
    ["Tenure", "MonthlyCharges", "SupportCalls",
     "CancellationRecorded"]
]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = (
    train_test_split(
        X_leaky,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
)

leaky_model = LogisticRegression(max_iter=1000)

leaky_model.fit(X_train_leaky, y_train_leaky)

leaky_predictions = leaky_model.predict(X_test_leaky)

leaky_accuracy = accuracy_score(
    y_test_leaky,
    leaky_predictions
)

print("\nIntentionally Incorrect Workflow")
print("--------------------------------")
print("Test Accuracy with Leaky Feature:",
      round(leaky_accuracy, 3))

print("\nImportant:")
print("The leaky feature reveals the target and must not be used.")
print("The two accuracy scores are illustrative, not a fair model comparison.")

Sample Dataset:
   Tenure  MonthlyCharges  SupportCalls  Churn  CancellationRecorded
0       2              90             5      1                     1
1       5              85             4      1                     1
2       8              80             4      1                     1
3      10              75             3      1                     1
4      12              70             3      1                     1

Training records: 30
Testing records: 10

Correct Workflow
----------------
Test Accuracy: 1.0

Intentionally Incorrect Workflow
--------------------------------
Test Accuracy with Leaky Feature: 1.0

Important:
The leaky feature reveals the target and must not be used.
The two accuracy scores are illustrative, not a fair model comparison.
